# 01 — Data Exploration & Preprocessing

This notebook walks through:
1. Loading sample resume and JD data
2. Text cleaning and preprocessing
3. Skill extraction using our taxonomy
4. Understanding what the preprocessing pipeline does at each step

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

# Load sample data
with open('../data/sample_resumes.json') as f:
    resumes = json.load(f)
with open('../data/sample_jds.json') as f:
    jds = json.load(f)

print(f'Loaded {len(resumes)} resumes and {len(jds)} job descriptions')
print(f'Resume names: {[r["name"] for r in resumes]}')

## 1. Raw Text Inspection

Let's look at what a raw resume looks like before any processing.

In [ ]:
# Inspect the first resume
sample_resume = resumes[0]
print(f"Name: {sample_resume['name']}")
print(f"Category: {sample_resume['category']}")
print(f"Text length: {len(sample_resume['text'])} chars")
print('---')
print(sample_resume['text'][:500])

## 2. Preprocessing Pipeline Step-by-Step

Let's see what happens at each stage of cleaning.

In [ ]:
from src.preprocessing import (
    remove_urls, remove_emails, remove_phone_numbers,
    remove_special_characters, normalize_whitespace,
    tokenize_and_lemmatize, clean_text, clean_text_for_skills
)

raw = sample_resume['text']

print('=== Step 1: Lowercase ===')
step1 = raw.lower()
print(step1[:200])

print('\n=== Step 2: Remove URLs ===')
step2 = remove_urls(step1)
print(step2[:200])

print('\n=== Step 3: Remove Emails ===')
step3 = remove_emails(step2)
print(step3[:200])

print('\n=== Step 4: Remove Special Characters ===')
step4 = remove_special_characters(step3)
print(step4[:200])

print('\n=== Step 5: Normalize Whitespace ===')
step5 = normalize_whitespace(step4)
print(step5[:200])

print('\n=== Step 6: Lemmatize + Remove Stopwords ===')
step6 = tokenize_and_lemmatize(step5)
print(step6[:200])

print('\n=== Full Pipeline (clean_text) ===')
full = clean_text(raw)
print(full[:200])

## 3. Skill Extraction

Using our curated taxonomy of ~200 skills to extract matches.

In [ ]:
from src.skill_extractor import SkillExtractor

extractor = SkillExtractor()

# Show taxonomy stats
print(f'Taxonomy categories: {list(extractor.taxonomy.keys())}')
print(f'Total known skills: {len(extractor.all_known_skills)}')
print()

In [ ]:
# Extract skills from Alice's resume
result = extractor.extract_skills(sample_resume['text'])

print(f"Taxonomy matches ({len(result['taxonomy_matches'])}):")
for s in sorted(result['taxonomy_matches']):
    print(f"  ✓ {s}")

print(f"\nNoun phrases ({len(result['noun_phrases'])}):")
for s in sorted(list(result['noun_phrases'])[:10]):
    print(f"  ? {s}")

print(f"\nCategory breakdown:")
for cat, skills in result['category_matches'].items():
    print(f"  {cat}: {skills}")

## 4. Compare Skills Across All Resumes

In [ ]:
import pandas as pd

rows = []
for resume in resumes:
    skills = extractor.extract_skill_set(resume['text'])
    rows.append({
        'Name': resume['name'],
        'Category': resume['category'],
        'Skill Count': len(skills),
        'Top Skills': ', '.join(sorted(skills)[:5])
    })

df = pd.DataFrame(rows)
df.sort_values('Skill Count', ascending=False)

## 5. Job Description Parsing

In [ ]:
from src.jd_parser import JDParser

parser = JDParser(skill_extractor=extractor)

for jd in jds:
    parsed = parser.parse(jd['text'], jd.get('title', ''))
    print(f"=== {parsed.title} ===")
    print(f"Required skills: {sorted(parsed.required_skills)}")
    print(f"Preferred skills: {sorted(parsed.preferred_skills)}")
    print(f"Experience: {parsed.experience_years}+ years")
    print(f"Seniority: {parsed.seniority_level}")
    print()